# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)
**Nama Mahasiswa:** Wahyu Andika
**NIM:** 202310370311075
**Username GitHub:** dikameo

---

Analisis eksplorasi **MedFact-Synth** dari data hasil cleaning Milestone 2 (`data/processed/medfact_clean.parquet`).

## Panduan Pengerjaan Milestone 3

1. **Sumber Data:** dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, serta DuckDB window function `OVER (...)` (Bagian 2).
3. **Visualisasi Interaktif:** 12 visualisasi interaktif Plotly (Bagian 3).
4. **Perumusan Insight:** 7 insight analitik berdasarkan hasil visualisasi (Bagian 4).

## Keterbatasan Dataset

Dataset tidak memiliki kolom tanggal maupun lokasi geografis, skemanya hanya teks klaim medis dan metadata PubMed. Karena itu:

- **Analisis spasial tidak dilakukan**: tidak ada data lokasi sama sekali.
- **Analisis temporal memakai proxy**: PubMed ID (PMID) bertambah seiring waktu artikel masuk ke PubMed, sehingga PMID yang lebih besar *cenderung* lebih baru. Ini **bukan tanggal asli**, hanya urutan kasar, dan diberi label proxy pada hasilnya.
- Visualisasi 6 pada template (pola spasial) digantikan oleh analisis korelasi multi-dimensi dan komposisi hierarkis (Visualisasi 10 dan 11).

In [1]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_GLOB = (RAW_DIR / 'data' / '*.parquet').as_posix()

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'medfact_clean.parquet'

if not CLEAN_PATH.exists():
    raise FileNotFoundError(f'{CLEAN_PATH} belum ada. Jalankan 02_data_cleaning.ipynb terlebih dahulu.')

lf = pl.scan_parquet(CLEAN_PATH)
n_rows = lf.select(pl.len()).collect(engine='streaming').item()
print(f'Jumlah baris data bersih: {n_rows:,}')

Jumlah baris data bersih: 1,497,952


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Karena tidak ada tanggal, sumbu waktu diganti **PMID** (proxy urutan). Teknik windowing yang dipakai:

- Polars `group_by_dynamic()` pada indeks integer: membagi PMID menjadi jendela 1.000.000 (`every='1000000i'`).
- Polars `rolling_mean()`: rata-rata bergerak 3 jendela untuk meredam fluktuasi.
- Polars `shift()`: selisih rata-rata label dengan jendela sebelumnya.
- DuckDB window function `AVG(...) OVER (...)` dan `LAG(...) OVER (...)` sebagai verifikasi silang hasil Polars.

In [2]:
pmid_ts = (
    lf.select(
        pl.col('claim_pmid').cast(pl.Int64).alias('pmid'),
        pl.col('synthetic_label').alias('label'),
        (pl.col('claim_potential') == 'support').cast(pl.Float64).alias('is_support'),
    )
    .sort('pmid')
    .collect(engine='streaming')
)

tren_pmid = (
    pmid_ts.group_by_dynamic('pmid', every='1000000i', closed='left')
    .agg(
        pl.len().alias('jumlah_baris'),
        pl.col('label').mean().alias('rata_label'),
        pl.col('is_support').mean().alias('proporsi_support'),
    )
    .with_columns(
        pl.col('rata_label').rolling_mean(window_size=3).alias('ma_3'),
        pl.col('rata_label').shift(1).alias('label_sebelumnya'),
    )
    .with_columns((pl.col('rata_label') - pl.col('label_sebelumnya')).alias('perubahan_label'))
    .rename({'pmid': 'pmid_mulai'})
)
tren_pmid

pmid_mulai,jumlah_baris,rata_label,proporsi_support,ma_3,label_sebelumnya,perubahan_label
i64,u32,f64,f64,f64,f64,f64
0,29471,0.221234,0.613959,null,null,null
1000000,45082,0.218446,0.610976,null,0.221234,-0.002788
2000000,53056,0.230078,0.615463,0.223253,0.218446,0.011631
3000000,41519,0.220333,0.611262,0.222952,0.230078,-0.009745
4000000,6378,0.235497,0.607557,0.228636,0.220333,0.015164
…,…,…,…,…,…,…
35000000,38278,0.128612,0.551309,0.130363,0.133337,-0.004725
36000000,36926,0.12544,0.541028,0.12913,0.128612,-0.003172
37000000,36642,0.120272,0.544812,0.124775,0.12544,-0.005168


In [3]:
# Verifikasi silang dengan window function DuckDB
tren_duckdb = duckdb.sql('''
    SELECT
        pmid_mulai,
        AVG(rata_label) OVER (ORDER BY pmid_mulai ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ma_3_duckdb,
        rata_label - LAG(rata_label) OVER (ORDER BY pmid_mulai) AS perubahan_duckdb
    FROM tren_pmid
    ORDER BY pmid_mulai
''').pl()

verifikasi = tren_pmid.join(tren_duckdb, on='pmid_mulai').select(
    'pmid_mulai', 'ma_3', 'ma_3_duckdb', 'perubahan_label', 'perubahan_duckdb'
)
# rolling_mean Polars bernilai null pada 2 jendela pertama, DuckDB menghitung dari data yang tersedia
cocok = verifikasi.drop_nulls('ma_3').select(((pl.col('ma_3') - pl.col('ma_3_duckdb')).abs() < 1e-9).all()).item()
print('Polars rolling_mean cocok dengan DuckDB window:', cocok)
verifikasi

Polars rolling_mean cocok dengan DuckDB window: True


pmid_mulai,ma_3,ma_3_duckdb,perubahan_label,perubahan_duckdb
i64,f64,f64,f64,f64
0,null,0.221234,null,null
1000000,null,0.21984,-0.002788,-0.002788
2000000,0.223253,0.223253,0.011631,0.011631
3000000,0.222952,0.222952,-0.009745,-0.009745
4000000,0.228636,0.228636,0.015164,0.015164
…,…,…,…,…
35000000,0.130363,0.130363,-0.004725,-0.004725
36000000,0.12913,0.12913,-0.003172,-0.003172
37000000,0.124775,0.124775,-0.005168,-0.005168


## Visualisasi 1: Distribusi `synthetic_label`

Skor kesesuaian antara klaim dan artikel sumber, dari -2 sampai 2.

In [4]:
label_counts = (
    lf.group_by('synthetic_label')
    .agg(pl.len().alias('jumlah'))
    .sort('synthetic_label')
    .collect(engine='streaming')
    .with_columns(
        pl.col('synthetic_label').cast(pl.Int8).cast(pl.Utf8).alias('label'),
        (pl.col('jumlah') / n_rows * 100).round(2).alias('persen'),
    )
)

fig1 = px.bar(
    label_counts, x='label', y='jumlah', text='persen',
    title='Distribusi synthetic_label',
    labels={'label': 'Label', 'jumlah': 'Jumlah baris', 'persen': 'Persen'},
)
fig1.update_traces(texttemplate='%{text}%')
fig1.show()
label_counts

synthetic_label,jumlah,label,persen
f64,u32,str,f64
-2.0,266666,"""-2""",17.8
-1.0,111817,"""-1""",7.46
0.0,545588,"""0""",36.42
1.0,228043,"""1""",15.22
2.0,345838,"""2""",23.09


## Visualisasi 2: Distribusi `claim_potential`

In [5]:
potential_counts = (
    lf.group_by('claim_potential')
    .agg(pl.len().alias('jumlah'))
    .sort('jumlah', descending=True)
    .collect(engine='streaming')
    .with_columns((pl.col('jumlah') / n_rows * 100).round(2).alias('persen'))
)

fig2 = px.pie(
    potential_counts, names='claim_potential', values='jumlah', hole=0.4,
    title='Distribusi claim_potential',
)
fig2.show()
potential_counts

claim_potential,jumlah,persen
str,u32,f64
"""support""",870840,58.14
"""refute""",627112,41.86


## Visualisasi 3: `claim_potential` vs `synthetic_label`

Apakah kategori potensi klaim berkorelasi dengan skor akhirnya?

In [6]:
cross_tab = (
    lf.group_by(['claim_potential', 'synthetic_label'])
    .agg(pl.len().alias('jumlah'))
    .sort(['claim_potential', 'synthetic_label'])
    .collect(engine='streaming')
    .with_columns(
        pl.col('synthetic_label').cast(pl.Int8).cast(pl.Utf8).alias('label'),
        (pl.col('jumlah') / pl.col('jumlah').sum().over('claim_potential') * 100).round(1).alias('persen_dalam_kategori'),
    )
)

fig3 = px.density_heatmap(
    cross_tab, x='label', y='claim_potential', z='persen_dalam_kategori',
    histfunc='sum', text_auto=True, color_continuous_scale='Blues',
    category_orders={'label': ['-2', '-1', '0', '1', '2']},
    title='Sebaran label dalam tiap kategori claim_potential (% per kategori)',
    labels={'label': 'Label', 'claim_potential': 'Kategori potensi klaim'},
)
fig3.show()
cross_tab

claim_potential,synthetic_label,jumlah,label,persen_dalam_kategori
str,f64,u32,str,f64
"""refute""",-2.0,184332,"""-2""",29.4
"""refute""",-1.0,65186,"""-1""",10.4
"""refute""",0.0,232050,"""0""",37.0
"""refute""",1.0,83825,"""1""",13.4
"""refute""",2.0,61719,"""2""",9.8
"""support""",-2.0,82334,"""-2""",9.5
"""support""",-1.0,46631,"""-1""",5.4
"""support""",0.0,313538,"""0""",36.0
"""support""",1.0,144218,"""1""",16.6


## Visualisasi 4 dan 5: Distribusi Panjang Teks

Panjang karakter `claim` dan `source`. Data dikelompokkan (binning) di Polars lebih dulu agar grafik interaktif tetap ringan, tidak mengirim 1,5 juta titik ke browser.

In [7]:
panjang = lf.select(
    pl.col('claim').str.len_chars().alias('panjang_claim'),
    pl.col('source').str.len_chars().alias('panjang_source'),
).collect(engine='streaming')


def histogram_biner(kolom, lebar, batas_atas):
    return (
        panjang.select(((pl.col(kolom).clip(upper_bound=batas_atas) // lebar) * lebar).alias('mulai'))
        .group_by('mulai').agg(pl.len().alias('jumlah')).sort('mulai')
    )


fig4 = px.bar(
    histogram_biner('panjang_claim', 20, 600), x='mulai', y='jumlah',
    title='Distribusi panjang teks claim (karakter, bin 20; nilai >600 digabung)',
    labels={'mulai': 'Panjang claim (karakter)', 'jumlah': 'Jumlah baris'},
)
fig4.show()

fig5 = px.bar(
    histogram_biner('panjang_source', 200, 6000), x='mulai', y='jumlah',
    title='Distribusi panjang teks source (karakter, bin 200; nilai >6000 digabung)',
    labels={'mulai': 'Panjang source (karakter)', 'jumlah': 'Jumlah baris'},
)
fig5.show()

panjang.describe()

statistic,panjang_claim,panjang_source
str,f64,f64
"""count""",1.497952e6,1.497952e6
"""null_count""",0.0,0.0
"""mean""",142.794284,1609.396825
"""std""",41.17133,529.652833
"""min""",28.0,10.0
"""25%""",112.0,1275.0
"""50%""",137.0,1622.0
"""75%""",169.0,1908.0
"""max""",437.0,14418.0


## Visualisasi 6: Artikel Sumber dengan Klaim Turunan Terbanyak

Satu `claim_pmid` dapat muncul pada banyak baris karena dipasangkan dengan banyak artikel `source`. Berikut 15 artikel yang paling banyak muncul.

In [8]:
klaim_per_pmid = lf.group_by('claim_pmid').agg(pl.len().alias('jumlah_klaim'))

top_pmid = klaim_per_pmid.sort('jumlah_klaim', descending=True).head(15).collect(engine='streaming')
statistik_pmid = klaim_per_pmid.select(
    pl.len().alias('jumlah_pmid'),
    pl.col('jumlah_klaim').mean().alias('rata_rata'),
    pl.col('jumlah_klaim').median().alias('median'),
    pl.col('jumlah_klaim').max().alias('maks'),
).collect(engine='streaming')

fig6 = px.bar(
    top_pmid, x='claim_pmid', y='jumlah_klaim',
    title='Top 15 claim_pmid dengan baris terbanyak',
    labels={'claim_pmid': 'PMID artikel', 'jumlah_klaim': 'Jumlah baris'},
)
fig6.update_xaxes(type='category')
fig6.show()
statistik_pmid

jumlah_pmid,rata_rata,median,maks
u32,f64,f64,u32
426440,3.512691,3.0,10


## Visualisasi 7: Proxy Temporal Berdasarkan PMID

**Peringatan metodologis:** PMID bukan tanggal. Grafik ini hanya menunjukkan **urutan relatif** artikel (10 kelompok kuantil nilai PMID, Kelompok 1 = PMID terkecil/cenderung paling lama), bukan garis waktu kalender. Tabel konversi PMID ke tanggal tidak tersedia pada dataset ini.

In [9]:
NAMA_KELOMPOK = [f'K{i:02d}' for i in range(1, 11)]

pmid_binned = (
    lf.select(pl.col('claim_pmid').cast(pl.Int64).alias('pmid'), pl.col('synthetic_label').cast(pl.Int8).cast(pl.Utf8).alias('label'))
    .collect(engine='streaming')
    .with_columns(pl.col('pmid').qcut(10, labels=NAMA_KELOMPOK).alias('kelompok_pmid'))
)

pmid_label = (
    pmid_binned.group_by(['kelompok_pmid', 'label']).agg(pl.len().alias('jumlah_baris'))
    .sort(['kelompok_pmid', 'label'])
)
pmid_rentang = pmid_binned.group_by('kelompok_pmid').agg(
    pl.col('pmid').min().alias('pmid_min'), pl.col('pmid').max().alias('pmid_max'), pl.len().alias('jumlah_baris')
).sort('pmid_min')

fig7 = px.bar(
    pmid_label.with_columns(pl.col('kelompok_pmid').cast(pl.Utf8)),
    x='kelompok_pmid', y='jumlah_baris', color='label', barmode='stack',
    category_orders={'label': ['-2', '-1', '0', '1', '2'], 'kelompok_pmid': NAMA_KELOMPOK},
    title='Baris per kelompok PMID dan label (proxy urutan, bukan tanggal asli)',
    labels={'kelompok_pmid': 'Kelompok PMID (K01 = PMID terkecil)', 'jumlah_baris': 'Jumlah baris', 'label': 'Label'},
)
fig7.show()
pmid_rentang

C:\Users\Huawei\AppData\Local\Temp\ipykernel_18400\341724836.py:6: DeprecationWarning: `qcut` is deprecated; use `bin_quantiles` or `bin_ranks` instead
  .with_columns(pl.col('pmid').qcut(10, labels=NAMA_KELOMPOK).alias('kelompok_pmid'))


kelompok_pmid,pmid_min,pmid_max,jumlah_baris
cat,i64,i64,u32
"""K01""",77,3531386,149799
"""K02""",3531428,8824240,149795
"""K03""",8824527,12100735,149797
"""K04""",12100763,17201267,149792
"""K05""",17201328,21075965,149794
"""K06""",21076015,24523447,149795
"""K07""",24523500,28097734,149794
"""K08""",28097883,31865049,149795
"""K09""",31865075,35699887,149795


## Visualisasi 8: Tren Rata-rata Label per Jendela PMID (Line Chart dengan Range Slider)

Hasil `group_by_dynamic` dan `rolling_mean` dari Bagian 2. Sumbu x adalah PMID (proxy waktu), bukan tanggal.

In [10]:
fig8 = go.Figure()
fig8.add_trace(go.Scatter(x=tren_pmid['pmid_mulai'], y=tren_pmid['rata_label'], mode='lines+markers', name='Rata-rata label'))
fig8.add_trace(go.Scatter(x=tren_pmid['pmid_mulai'], y=tren_pmid['ma_3'], mode='lines', name='Rata-rata bergerak (3 jendela)', line=dict(dash='dash')))
fig8.update_layout(
    title='Tren rata-rata synthetic_label per jendela 1 juta PMID (proxy waktu)',
    xaxis=dict(title='PMID awal jendela', rangeslider=dict(visible=True)),
    yaxis_title='Rata-rata synthetic_label',
)
fig8.show()

## Visualisasi 9: Distribusi dan Outlier Panjang `source` per Label (Box Plot)

Box plot dihitung dari sampel acak 100.000 baris (seed tetap) agar ringan dikirim ke browser.

In [11]:
fitur = lf.select(
    pl.col('claim_pmid').cast(pl.Int64).alias('pmid'),
    pl.col('synthetic_label'),
    (pl.col('claim_potential') == 'support').cast(pl.Int8).alias('is_support'),
    pl.col('claim').str.len_chars().alias('panjang_claim'),
    pl.col('source').str.len_chars().alias('panjang_source'),
).collect(engine='streaming')

sampel = fitur.sample(100_000, seed=42).with_columns(pl.col('synthetic_label').cast(pl.Int8).cast(pl.Utf8).alias('label'))

fig9 = px.box(
    sampel, x='label', y='panjang_source', color='label',
    category_orders={'label': ['-2', '-1', '0', '1', '2']},
    title='Panjang source per label (sampel 100.000 baris)',
    labels={'label': 'Label', 'panjang_source': 'Panjang source (karakter)'},
)
fig9.show()
sampel.group_by('label').agg(
    pl.col('panjang_source').median().alias('median'),
    pl.col('panjang_source').quantile(0.99).alias('p99'),
    pl.col('panjang_source').max().alias('maks'),
).sort('label')

label,median,p99,maks
str,f64,f64,u32
"""-1""",1626.0,3068.0,9946
"""-2""",1598.0,2968.0,9386
"""0""",1663.0,3023.0,13151
"""1""",1627.0,3024.0,6621
"""2""",1559.0,2925.0,7587


## Visualisasi 10: Korelasi Antar-Variabel Numerik (Heatmap)

Korelasi Pearson dihitung pada seluruh baris data bersih. `is_support` bernilai 1 untuk `claim_potential = support`.

In [12]:
korelasi = fitur.select('pmid', 'synthetic_label', 'is_support', 'panjang_claim', 'panjang_source').corr()
kolom_korelasi = korelasi.columns

fig10 = px.imshow(
    korelasi.to_numpy().round(3), x=kolom_korelasi, y=kolom_korelasi,
    text_auto=True, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Matriks korelasi Pearson antar-variabel numerik',
)
fig10.show()
korelasi

pmid,synthetic_label,is_support,panjang_claim,panjang_source
f64,f64,f64,f64,f64
1.0,-0.024289,-0.048097,0.010359,0.117287
-0.024289,1.0,0.341745,0.200232,-0.028202
-0.048097,0.341745,1.0,0.556042,-0.003022
0.010359,0.200232,0.556042,1.0,0.081617
0.117287,-0.028202,-0.003022,0.081617,1.0


## Visualisasi 11: Komposisi Kategori Hierarkis (Treemap)

Komposisi `claim_potential` yang dipecah per `synthetic_label`.

In [13]:
fig11 = px.treemap(
    cross_tab.with_columns(pl.col('claim_potential').alias('kategori')),
    path=[px.Constant('Semua baris'), 'kategori', 'label'], values='jumlah',
    title='Komposisi baris: claim_potential > synthetic_label',
)
fig11.show()

## Visualisasi 12: Perbandingan Kelompok PMID, Proporsi Label (Heatmap)

Membandingkan komposisi label (% dalam tiap kelompok) antar 10 kelompok kuantil PMID. Ini analisis perbandingan kelompok, bukan siklus musiman asli (tidak ada tanggal).

In [14]:
pmid_persen = pmid_label.with_columns(
    pl.col('kelompok_pmid').cast(pl.Utf8),
    (pl.col('jumlah_baris') / pl.col('jumlah_baris').sum().over('kelompok_pmid') * 100).round(2).alias('persen'),
)

fig12 = px.density_heatmap(
    pmid_persen, x='kelompok_pmid', y='label', z='persen', histfunc='sum', text_auto=True,
    color_continuous_scale='Viridis',
    category_orders={'kelompok_pmid': NAMA_KELOMPOK, 'label': ['-2', '-1', '0', '1', '2']},
    title='Komposisi label per kelompok PMID (% dalam kelompok, proxy urutan)',
    labels={'kelompok_pmid': 'Kelompok PMID (K01 = terkecil)', 'label': 'Label'},
)
fig12.show()
pmid_persen.pivot(on='label', index='kelompok_pmid', values='persen').sort('kelompok_pmid')

kelompok_pmid,-1,-2,0,1,2
str,f64,f64,f64,f64,f64
"""K01""",7.14,17.3,35.97,14.88,24.71
"""K02""",7.07,17.33,35.98,14.8,24.83
"""K03""",7.35,17.44,35.8,15.4,24.02
"""K04""",7.4,17.64,35.98,15.35,23.62
"""K05""",7.43,17.71,36.18,15.24,23.44
"""K06""",7.57,17.91,36.28,15.32,22.93
"""K07""",7.42,17.84,36.7,15.21,22.83
"""K08""",7.57,18.07,36.94,15.21,22.21
"""K09""",7.85,18.27,37.03,15.38,21.46


## Angka Pendukung Insight

Cell ini menghitung angka dari data bersih yang dipakai pada insight di bawah.

In [15]:
print('Label dominan dalam tiap kategori claim_potential:')
dominan = (
    cross_tab.sort('persen_dalam_kategori', descending=True)
    .group_by('claim_potential', maintain_order=True).first()
)
for b in dominan.iter_rows(named=True):
    print(f"  {b['claim_potential']:<8} -> label {b['label']} ({b['persen_dalam_kategori']}%)")

print()
print('Artikel dengan baris terbanyak:', top_pmid.row(0, named=True))
print(statistik_pmid)

print()
print('Median panjang teks (karakter):')
print(panjang.select(pl.col('panjang_claim').median(), pl.col('panjang_source').median()))

print()
print('Rata-rata label per jendela PMID (awal, akhir, selisih):')
print(tren_pmid.select(
    pl.col('rata_label').first().round(4).alias('awal'),
    pl.col('rata_label').last().round(4).alias('akhir'),
    (pl.col('rata_label').last() - pl.col('rata_label').first()).round(4).alias('selisih'),
))
print()
print('Korelasi dengan synthetic_label:')
print(korelasi.with_columns(pl.Series('variabel', kolom_korelasi)).select('variabel', 'synthetic_label').filter(pl.col('variabel') != 'synthetic_label'))

Label dominan dalam tiap kategori claim_potential:
  refute   -> label 0 (37.0%)
  support  -> label 0 (36.0%)

Artikel dengan baris terbanyak: {'claim_pmid': '16261280', 'jumlah_klaim': 10}
shape: (1, 4)
┌─────────────┬───────────┬────────┬──────┐
│ jumlah_pmid ┆ rata_rata ┆ median ┆ maks │
│ ---         ┆ ---       ┆ ---    ┆ ---  │
│ u32         ┆ f64       ┆ f64    ┆ u32  │
╞═════════════╪═══════════╪════════╪══════╡
│ 426440      ┆ 3.512691  ┆ 3.0    ┆ 10   │
└─────────────┴───────────┴────────┴──────┘

Median panjang teks (karakter):
shape: (1, 2)
┌───────────────┬────────────────┐
│ panjang_claim ┆ panjang_source │
│ ---           ┆ ---            │
│ f64           ┆ f64            │
╞═══════════════╪════════════════╡
│ 137.0         ┆ 1622.0         │
└───────────────┴────────────────┘

Rata-rata label per jendela PMID (awal, akhir, selisih):
shape: (1, 3)
┌────────┬────────┬─────────┐
│ awal   ┆ akhir  ┆ selisih │
│ ---    ┆ ---    ┆ ---     │
│ f64    ┆ f64    ┆ f64     │
╞══

## Ringkasan Insight

Persentase di bawah dihitung dari 1.497.981 baris pada pemeriksaan awal; pembersihan hanya membuang ±29 baris sehingga selisihnya tidak material. Angka yang bergantung pada tabel silang dan top PMID dicetak otomatis oleh cell *Angka Pendukung Insight* di atas.

1. **Distribusi label tidak seimbang** (Visualisasi 1). Label `0` (netral) paling banyak dengan 545.603 baris (36,4%), diikuti label `2` (23,1%), `-2` (17,8%), `1` (15,2%), dan `-1` paling sedikit (7,5%). Implikasi: jika label dipakai untuk melatih model klasifikasi, kelas `-1` kurang terwakili sehingga perlu class weighting atau resampling.
2. **`claim_potential` hanya dua kategori** (Visualisasi 2). `support` sebanyak 870.858 baris (58,1%) dan `refute` 627.123 baris (41,9%). Implikasi: dataset condong ke klaim yang didukung, tetapi cukup seimbang untuk klasifikasi biner tanpa penanganan khusus.
3. **`claim_potential` berhubungan dengan `synthetic_label`** (Visualisasi 3). Heatmap menampilkan persentase tiap label dalam kategori `support` dan `refute`; label dominan masing-masing kategori dicetak pada cell *Angka Pendukung Insight*. Implikasi: jika sebaran label berbeda jelas antar kategori, `claim_potential` layak menjadi fitur prediktif, tetapi juga berisiko menjadi kebocoran informasi bila dipakai untuk memprediksi label.
4. **Panjang `source` jauh lebih bervariasi daripada `claim`** (Visualisasi 4 dan 5). `claim` berkisar 28 s.d. 437 karakter dengan median 137, sedangkan `source` berkisar 10 s.d. 14.418 karakter dengan median 1.622 dan ekor kanan panjang. Implikasi: perlu strategi truncation atau padding terpisah untuk tiap kolom pada tahap NLP.
5. **Satu artikel muncul pada banyak baris** (Visualisasi 6). Terdapat 426.440 `claim_pmid` unik untuk 1.497.981 baris, rata-rata ±3,5 baris per artikel, dan sebaran di ekor atas dicetak pada cell pendukung. Implikasi: pemisahan data latih dan uji harus dilakukan per `claim_pmid` agar klaim dari artikel yang sama tidak bocor ke kedua sisi.
6. **Tren proxy temporal: rata-rata label menurun pada PMID besar** (Visualisasi 8 dan 12, hasil `group_by_dynamic`, `rolling_mean`, `shift`, dan DuckDB `OVER`). Rata-rata `synthetic_label` turun dari 0,221 pada jendela PMID terkecil menjadi 0,117 pada jendela terbesar (selisih -0,105), dan proporsi label `2` turun dari 24,7% (K01) menjadi 20,8% (K10). Implikasi: distribusi label bergeser terhadap urutan PMID, jadi pembagian data latih/uji acak dapat menyembunyikan pergeseran ini; perlu uji berbasis urutan PMID. Ini hanya proxy karena PMID bukan tanggal.
7. **Korelasi label paling kuat dengan `claim_potential`, lalu panjang `claim`** (Visualisasi 10). Korelasi dengan `synthetic_label` adalah 0,34 untuk `is_support`, 0,20 untuk `panjang_claim`, serta hampir nol untuk `panjang_source` (-0,03) dan PMID (-0,02). `panjang_claim` juga berkorelasi 0,56 dengan `is_support`. Implikasi: panjang klaim dapat menjadi jalan pintas (shortcut) bagi model dan perlu diwaspadai pada evaluasi.
8. **Panjang `source` tidak membedakan label** (Visualisasi 9 dan 11). Median panjang `source` per label berkisar sempit, 1.559 s.d. 1.663 karakter, dan outlier panjang (hingga 13.151 karakter pada label `0`) ada di semua label. Komposisi treemap menunjukkan `support` didominasi label `2` dan `0`, sedangkan `refute` didominasi label `0` dan `-2`. Implikasi: truncation `source` pada panjang tertentu aman dari sisi bias label, tetapi pemisahan per `claim_potential` tetap relevan.
9. **Keterbatasan dimensi waktu dan ruang** *(temuan metodologis, bukan insight dari data)*. Dataset tidak punya tanggal maupun lokasi; seluruh analisis temporal memakai PMID sebagai proxy dan analisis spasial tidak dapat dilakukan.

### Keterbatasan Analisis

- **Tidak ada analisis spasial**: dataset tidak memuat informasi lokasi geografis.
- **Analisis temporal bersifat proxy**: memakai urutan kasar PMID, bukan tanggal kalender.
- **Dataset bukan dataset Indonesia**, sehingga tidak memenuhi syarat "Dataset Indonesia" pada spesifikasi tugas di README. Hal ini dicatat secara eksplisit sebagai kejujuran akademik.